<a href="https://colab.research.google.com/github/silvatria/pembelajaran-mesin-254107023001/blob/main/JS03_TL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Langkah 1 - Import & Load
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("data.csv")

# Langkah 2 - Pisahkan variabel
drop_cols = [c for c in ["id", "Unnamed: 32"] if c in df.columns]
df = df.drop(columns=drop_cols)

y = LabelEncoder().fit_transform(df["diagnosis"])  # M=1, B=0
X = df.drop(columns=["diagnosis"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

# Langkah 3-5 - Pipeline: Standardisasi + Seleksi Fitur + Model
results = {}
for k in range(1, X.shape[1] + 1):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("sel", SelectKBest(score_func=f_classif, k=k)),
        ("clf", LogisticRegression(max_iter=1000))
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results[k] = accuracy_score(y_test, pred)

# Langkah 7 - Cari k terbaik
best_k = max(results, key=results.get)
print(f"Best k: {best_k}, Accuracy: {results[best_k]:.4f}")

# Fit ulang dengan best_k buat lihat fitur yang kepilih
final_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("sel", SelectKBest(score_func=f_classif, k=best_k)),
    ("clf", LogisticRegression(max_iter=1000))
])
final_pipe.fit(X_train, y_train)
mask = final_pipe.named_steps["sel"].get_support()
selected_features = X.columns[mask]
print("Fitur terpilih:", list(selected_features))
print(classification_report(y_test, final_pipe.predict(X_test)))

# Jumlah fitur terbaik: 14 fitur (k=14), dipilih berdasarkan hasil SelectKBest dengan skor ANOVA F-test (f_classif) yang menghasilkan akurasi tertinggi (98%) pada data uji :
# radius_mean
# perimeter_mean
# area_mean
# compactness_mean
# concavity_mean
# concave points_mean
# radius_se
# perimeter_se
# radius_worst
# perimeter_worst
# area_worst
# compactness_worst
# concavity_worst
# concave points_worst

Best k: 14, Accuracy: 0.9825
Fitur terpilih: ['radius_mean', 'perimeter_mean', 'area_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'radius_se', 'perimeter_se', 'radius_worst', 'perimeter_worst', 'area_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst']
              precision    recall  f1-score   support

           0       0.97      1.00      0.99        72
           1       1.00      0.95      0.98        42

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

